## Logistic Regression

In [1]:
import numpy as np

**Q**: Using NumPy, write a function to calculate the logistic function and its elementwise derivative ($\sigma^\prime (\cdot)$).

**A**: See the function `logistic_prime` below

---

In [6]:
def logistic(x: np.ndarray):
    return 1 / (1 + np.exp(-x))

def logistic_prime(x: np.ndarray):
    y = logistic(x)
    return y * (1 - y)

**Q**: How is the likelihood of a logistic regression model defined for a single data point?

**A**:
$$L(\vec{\beta} \mid x^{(i)}, y_i) = \hat{y}_i^{y_i}(1-\hat{y}_i)^{1-y_i},$$ 
where
$$\hat{y}_i = f \big( x^{(i)} \cdot \vec{\beta} \big),$$
and $f$ is the logistic function. 

---

**Q**: Using NumPy, write a function to compute the negative log likelihood for a single data point

**A**: See the function `negative_log_likelihood` below

---

In [7]:
def negative_log_likelihood(y, y_hat):
    if y == 1:
        return - np.log(y_hat)
    elif y == 0:
        return - np.log(1 - y_hat)

**Q**: In logistic regression, what is the partial derivative of the negative log likelihood for a single data point with respect to $\beta_j$?

**A**:  $$\frac{\partial l}{\partial \beta_j} = - (y_i - \hat{y}_i) \cdot x^{(i)}_j$$

**Explanation**: We have:
$$
\begin{align*}
\frac{\partial l}{\partial \beta_j} &= - \frac{y_i}{\hat{y}_i} \cdot \frac{\partial \hat{y}_i}{\partial \beta_j} - \frac{1 - y_i}{1 - \hat{y}_i} \cdot \frac{\partial (1 - \hat{y}_i)}{\partial \beta_j} \\[4pt]
&= - \frac{y_i}{\hat{y}_i} \cdot \frac{\partial \hat{y}_i}{\partial \beta_j} + \frac{1 - y_i}{1 - \hat{y}_i} \cdot \frac{\partial \hat{y}_i}{\partial \beta_j} \\[4pt]
&= - \frac{y_i}{\hat{y}_i} \cdot \hat{y}_i (1 - \hat{y}_i) \cdot x^{(i)}_j + \frac{1 - y_i}{1 - \hat{y}_i} \cdot \hat{y}_i (1 - \hat{y}_i) \cdot x^{(i)}_j \\[4pt]
&= - y_i (1 - \hat{y}_i) \cdot x^{(i)}_j + (1 - y_i) \cdot \hat{y}_i \cdot x^{(i)}_j \\[4pt]
&= (\hat{y}_i - y_i) \cdot x^{(i)}_j \\[4pt]
&= - (y_i - \hat{y}_i) \cdot x^{(i)}_j
\end{align*}
$$

---

**Q**: In logistic regression, write a function to compute the gradient of the negative log likelihood function for a single data point using NumPy, given $y$, $\hat{y}$, and $x$

**A**: See the function `neg_log_likelihood_grad` below

---

In [11]:
def neg_log_likelihood_grad(
    y: float, 
    y_hat: float, 
    x: np.ndarray
) -> np.ndarray:
    return - (y - y_hat) * x

**Q**: Using NumPy, write a function to find the optimal parameters for logistic regression by minimizing the negative log likelihood loss using mini-batch gradient descent. Assume you have the functions `neg_log_likelihood_grad`, `logistic`, and `neg_log_likelihood_loss` at your disposal.

**A**: See the function `logistic_regression_fit` below

---

**Q**: Using NumPy, write a function to compute the negative log likelihood loss in logistic regression.

**A**: See the function `neg_log_likelihood_loss` below

---

**Q**: Using NumPy, write a function to compute the gradient of the negative log likelihood loss in logistic regression.

**A**: See the function `neg_log_likelihood_grad` below

---

In [17]:
def neg_log_likelihood_loss(
    y: np.ndarray,
    y_hat: np.ndarray
) -> float:
    return -np.mean(
        y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat)
    )

def neg_log_likelihood_grad(
    y: np.ndarray, 
    y_hat: np.ndarray, 
    X: np.ndarray
) -> np.ndarray:
    return - np.mean(X * (y - y_hat).reshape(-1, 1), axis=0)

def logistic_regression_fit(
    X: np.ndarray,
    y: np.ndarray,
    n_iters: int = 100,
    batch_size: int=32,
    lr: float = 0.01
) -> np.ndarray:
    assert X.shape[0] == y.shape[0], "X and y should have the same number of rows"

    n = X.shape[0]
    n_batches = n // batch_size if n % batch_size == 0 else 1 + n // batch_size

    params = np.random.randn(X.shape[1])

    for i in range(n_iters):
        total_avg_loss = 0
        for batch_idx in range(0, n, batch_size):
            X_batch = X[batch_idx: batch_idx + batch_size]
            y_batch = y[batch_idx: batch_idx + batch_size]

            raw_preds = X_batch @ params
            preds = logistic(raw_preds)
            loss = neg_log_likelihood_loss(y_batch, preds)

            grad = neg_log_likelihood_grad(y_batch, preds, X_batch)

            params -= lr*grad

            total_avg_loss += loss

        total_avg_loss = total_avg_loss / n_batches
        print(f"loss for iteration {i+1}: ", total_avg_loss)

    return params